# Notebook 3: Evaluate & Compare Models

Compare base models against fine-tuned models using ROUGE and qualitative analysis.

**This notebook:**
1. Loads base and fine-tuned models
2. Runs them on a held-out test set
3. Computes ROUGE metrics
4. Generates a side-by-side comparison report

In [ ]:
!pip install transformers torch rouge-score datasets accelerate -q

In [ ]:
import json
import time
from pathlib import Path
from transformers import pipeline
from rouge_score import rouge_scorer

MODELS_TO_COMPARE = {
    # Add your fine-tuned model paths here (Google Drive or HuggingFace)
    "bart-large-cnn (base)": "facebook/bart-large-cnn",
    "distilbart-cnn (base)": "sshleifer/distilbart-cnn-6-6",
    # "cyber-bart (finetuned)": "/content/drive/MyDrive/model-lab/models/cyber-bart-summarizer",
    # "cyber-distilbart (finetuned)": "/content/drive/MyDrive/model-lab/models/cyber-distilbart",
}

print("Configure MODELS_TO_COMPARE above with your fine-tuned model paths.")

## Load Test Data

In [ ]:
# Test data - different from training data
TEST_DATA = [
    {
        "article": "The BlackCat ransomware operation has shifted its tactics to deploy a new variant called Scattered Spider, combining ransomware deployment with data theft and extortion. The FBI reported that Scattered Spider actors have infiltrated at least 15 large organizations in the US and UK, leveraging sophisticated social engineering techniques including vishing and SIM swapping to gain initial access. Once inside, the actors move laterally using stolen credentials and deploy BlackCat ransomware while simultaneously exfiltrating sensitive data for double extortion. Organizations are advised to enforce phishing-resistant MFA and implement least-privilege access controls.",
        "reference": "BlackCat ransomware operators are using a new variant called Scattered Spider, combining ransomware with data theft and double extortion. The FBI reports infiltration of 15 large organizations in the US and UK using social engineering, vishing, and SIM swapping. Organizations should enforce phishing-resistant MFA and least-privilege controls."
    },
    {
        "article": "Apple has released emergency security updates for iOS, iPadOS, and macOS to patch two zero-day vulnerabilities that were actively exploited in the wild. CVE-2024-23225 affects the Kernel, allowing attackers to bypass kernel memory protections, while CVE-2024-23296 affects WebRTC, enabling malicious applications to access sensitive user data. Apple confirmed that both vulnerabilities may have been exploited against targeted individuals. The updates are available for iPhone 8 and later, iPad Pro, and macOS Ventura and later. Users are strongly encouraged to update immediately through Settings > General > Software Update.",
        "reference": "Apple released emergency patches for two actively exploited zero-days: CVE-2024-23225 (Kernel memory protection bypass) and CVE-2024-23296 (WebRTC data access). Both may have been used in targeted attacks. Updates are available for iOS, iPadOS, and macOS, and users should update immediately."
    },
    {
        "article": "The Linux Foundation announced the formation of the Open Source Security Foundation to address the growing challenge of securing open source software supply chains. The initiative brings together major technology companies including Google, Microsoft, Amazon, and IBM to fund security audits, develop new scanning tools, and establish best practices for open source maintenance. The foundation will focus on identifying the most critical open source projects, providing resources for security hardening, and creating a vulnerability disclosure framework tailored to the open source ecosystem.",
        "reference": "The Linux Foundation formed the Open Source Security Foundation to secure open source supply chains, with participation from Google, Microsoft, Amazon, and IBM. The initiative will fund security audits, develop scanning tools, and establish vulnerability disclosure practices for critical open source projects."
    },
    {
        "article": "Kaspersky researchers discovered a new advanced persistent threat group they named GoldenJackal, targeting diplomatic and government organizations in the Middle East and South Asia. The group uses custom toolsets including USB stealers, file exfiltration tools, and credential harvesters. GoldenJackal's primary infection vector is compromised USB drives, which they use to air-gap jump into isolated networks. The campaign has been active since 2019 and has compromised organizations in at least five countries. The tools demonstrate a high level of sophistication, suggesting state sponsorship.",
        "reference": "Kaspersky discovered GoldenJackal, an APT group targeting diplomatic and government organizations in the Middle East and South Asia. The group uses custom USB-based tools to jump air gaps and exfiltrate data. Active since 2019 across five countries, the sophistication suggests state sponsorship."
    },
    {
        "article": "The SolarWinds supply chain attack that was disclosed in December 2020 continues to reveal new dimensions of compromise. A joint advisory from CISA, the FBI, and NSA confirmed that the Russian APT29 group modified SolarWinds Orion network management software updates to insert a backdoor called Sunburst. The compromise affected approximately 18,000 organizations that downloaded the tainted update, though the threat actor actually exploited only about 100 of those for further operations. New analysis shows the attackers maintained persistent access to victim networks for an average of 14 months before detection, using sophisticated stealth techniques including custom malware and encrypted command channels.",
        "reference": "The SolarWinds attack by Russian APT29 involved inserting the Sunburst backdoor into Orion software updates, affecting 18,000 organizations with about 100 actively exploited. New analysis shows attackers maintained access for an average of 14 months using stealth techniques and encrypted communications."
    },
]

DATA_SOURCE = "built-in"  # Change to "drive" or "upload" for custom test data

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    test_data = []
    with open("/content/drive/MyDrive/model-lab/data/test.jsonl") as f:
        for line in f:
            if line.strip():
                test_data.append(json.loads(line))
    TEST_DATA = test_data
elif DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    test_data = []
    with open(fname) as f:
        for line in f:
            if line.strip():
                test_data.append(json.loads(line))
    TEST_DATA = test_data

print(f"Test set: {len(TEST_DATA)} articles")

## Run Evaluation

In [ ]:
import gc, torch

all_results = {}
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

for model_label, model_path in MODELS_TO_COMPARE.items():
    print(f"\n{'='*70}")
    print(f"Evaluating: {model_label}")
    print(f"Path: {model_path}")
    
    summarizer = pipeline("summarization", model=model_path, device_map="auto")
    
    model_results = []
    r1_scores, r2_scores, rl_scores = [], [], []
    times = []
    
    for i, sample in enumerate(TEST_DATA):
        start = time.time()
        try:
            output = summarizer(
                sample["article"],
                max_length=150,
                min_length=30,
                do_sample=False,
            )
            prediction = output[0]["summary_text"]
        except Exception as e:
            prediction = f"ERROR: {e}"
        infer_time = time.time() - start
        
        scores = scorer.score(sample["reference"], prediction)
        r1_scores.append(scores["rouge1"].fmeasure)
        r2_scores.append(scores["rouge2"].fmeasure)
        rl_scores.append(scores["rougeL"].fmeasure)
        times.append(infer_time)
        
        model_results.append({
            "article_id": i,
            "reference": sample["reference"],
            "prediction": prediction,
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure,
            "inference_time": infer_time,
        })
        print(f"  Article {i}: R1={scores['rouge1'].fmeasure:.4f} R2={scores['rouge2'].fmeasure:.4f} RL={scores['rougeL'].fmeasure:.4f} ({infer_time:.1f}s)")
    
    all_results[model_label] = {
        "model_path": model_path,
        "rouge1": sum(r1_scores) / len(r1_scores),
        "rouge2": sum(r2_scores) / len(r2_scores),
        "rougeL": sum(rl_scores) / len(rl_scores),
        "avg_inference_time": sum(times) / len(times),
        "predictions": model_results,
    }
    
    del summarizer
    gc.collect()
    torch.cuda.empty_cache()

## Comparison Table

In [ ]:
print(f"{'Model':<35} {'ROUGE-1':>10} {'ROUGE-2':>10} {'ROUGE-L':>10} {'Avg Time':>10}")
print("=" * 80)

for label, data in all_results.items():
    print(f"{label:<35} {data['rouge1']:>10.4f} {data['rouge2']:>10.4f} {data['rougeL']:>10.4f} {data['avg_inference_time']:>9.2f}s")n
best_r1 = max(all_results.items(), key=lambda x: x[1]["rouge1"])
best_rl = max(all_results.items(), key=lambda x: x[1]["rougeL"])
best_time = min(all_results.items(), key=lambda x: x[1]["avg_inference_time"])
print(f"\nBest ROUGE-1: {best_r1[0]} ({best_r1[1]['rouge1']:.4f})")
print(f"Best ROUGE-L: {best_rl[0]} ({best_rl[1]['rougeL']:.4f})")
print(f"Fastest: {best_time[0]} ({best_time[1]['avg_inference_time']:.2f}s)")

## Side-by-Side Summaries

In [ ]:
for i, sample in enumerate(TEST_DATA):
    print(f"\n{'='*80}")
    print(f"ARTICLE {i+1}: {sample['article'][:150]}...")
    print(f"REFERENCE: {sample['reference']}")
    print(f"{'-'*80}")
    
    for label, data in all_results.items():
        pred = data["predictions"][i]["prediction"]
        r1 = data["predictions"][i]["rouge1"]
        print(f"  {label}:")
        print(f"    {pred}")
        print(f"    [ROUGE-1: {r1:.4f}]")
    print()

## Save Results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_dir = Path('/content/drive/MyDrive/model-lab/evaluations')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "evaluation_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print(f"Results saved to {output_dir / 'evaluation_results.json'}")